# 01 · KITTI-360 supplement (one drive per run)

Prepares what the rest of OCGF needs for one KITTI-360 drive:

- **scans** with each point's laser ring (computed on the raw scan order), cropped to the BEV region, in compact shards;
- **images**: from the Kaggle starter for drive 0000, otherwise from the official `image_00` zip, stored in shards;
- **velodyne/image timestamps** and a **frame index**: usable frames, kept frames, frames sampled every 5 m;
- a **quality report** (missing frames, alignment, ring sanity, LiDAR drawn on images) and, for drive 0000, a **dev sample**.

## Paper drives (split `kitti360_paper_v1`, decided 2026-10-18)

| drive | role | keep frames | window rule |
|---|---|---|---|
| 0000 | train (prepared first as `dev`) | every frame | yes (prepared before the rule) |
| 0002, 0004, 0005 | train | every 2.5 m | no |
| 0006 | val | every 2.5 m | yes |
| 0009, 0010 | test | every 2.5 m | yes |

Only the kept frames are read from the remote zips, about a quarter of each zip. Run the test drives first.

## Before you run

1. **Settings → Accelerator:** None (CPU is enough). **Internet:** On.
2. **Add Input:** the KITTI-360 starter (calibration), your private `kitti360-poses` (all drives), `kitti360-3d-bboxes`,
   `kitti360-2d-semantics`, and for val/test drives the accumulated 3D semantics (`greatgamedota/kitti360-3d-semantics`).
3. Set `SEQUENCE`, `ROLE` and `KEEP_EVERY_M` below. Download links come from `configs/data_sources/kitti360.yaml`.
4. For a quick trial set `MAX_FRAMES = 200`. For the real run set `MAX_FRAMES = None` and **Save Version → Save & Run All**.

In [ ]:
from pathlib import Path
import time

REPO_URL     = "https://github.com/GNaveenkumar614/OCGF.git"
GIT_REF      = "p0/skeleton-geometry"      # branch, tag or commit (main once merged); pin a commit for the final run
SEQUENCE     = "2013_05_28_drive_0009_sync"
ROLE         = "test"                      # dev (drive 0000 only) | train | val | test, see the table above
KEEP_EVERY_M = 2.5                         # None keeps every frame (drive 0000); 2.5 for the other paper drives
INPUT_ROOT   = Path("/kaggle/input")
OUT_DIR      = Path("/kaggle/working/kitti360_supplement")   # kept as the notebook output
SCRATCH      = Path("/kaggle/temp")                          # temporary, not kept
MODE         = "auto"   # how to read remote zips: auto | download | remote
MAX_FRAMES   = None     # e.g. 200 for a quick trial; None for the full drive
DEV_FRAMES   = 0        # 50 for drive 0000 only

CODE_DIR = SCRATCH / "OCGF"
TIMINGS = {}

## Code

A fresh clone of the repo at `GIT_REF`, installed in editable mode.

In [ ]:
import shutil, subprocess, sys

def sh(cmd):
    print("$", cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True)

SCRATCH.mkdir(parents=True, exist_ok=True)
if CODE_DIR.exists():
    shutil.rmtree(CODE_DIR)            # fresh clone, so the run matches GIT_REF exactly
sh(f'git clone -q {REPO_URL} "{CODE_DIR}"')
sh(f'git -C "{CODE_DIR}" checkout -q {GIT_REF}')
sh(f'git -C "{CODE_DIR}" log -1 --date=short --format="code at commit %h (%cd): %s"')
sh(f'{sys.executable} -m pip install -q -e "{CODE_DIR}"')
sys.path.insert(0, str(CODE_DIR))      # the editable install is only seen by new processes

from ocgf.utils.provenance import environment
print(environment())

## Inputs

Finds every component under `/kaggle/input`; if one is missing it prints the folder tree and stops.

In [ ]:
from ocgf.datasets import prepare_kitti360 as prep

cfg = prep.PrepConfig(sequence=SEQUENCE, input_root=INPUT_ROOT, out_dir=OUT_DIR, scratch_dir=SCRATCH,
                      sources=CODE_DIR / "configs" / "data_sources" / "kitti360.yaml", role=ROLE,
                      keep_every_m=KEEP_EVERY_M, mode=MODE, max_frames=MAX_FRAMES, dev_frames=DEV_FRAMES)
sources = prep.load_sources(cfg.sources, SEQUENCE)

t = time.time()
found = prep.stage_inputs(cfg)
TIMINGS["inputs"] = round(time.time() - t, 1)

## Sources

Opens the velodyne zip and the images (starter folder or `image_00` zip) and extracts both timestamp files. With `KEEP_EVERY_M` set, `auto` reads the zips remotely, fetching only the kept frames.

In [ ]:
t = time.time()
src = prep.stage_sources(cfg, found, sources)
TIMINGS["sources"] = round(time.time() - t, 1)

## Frame selection

Usable = image, scan, pose, ≥ 50 m of future path, image/scan aligned, and (val/test/dev) inside a semantic window. Kept = every frame, or one usable frame every `KEEP_EVERY_M` metres.

In [ ]:
t = time.time()
index = prep.stage_select(cfg, found, src)
kept = index["frame"][index["kept"]]
TIMINGS["select"] = round(time.time() - t, 1)

## Scans

For every kept scan: ring index on the raw point order, crop to x ∈ [−10, 80) m, y ∈ [−40, 40) m, encode (8 bytes per point), write to shards.

In [ ]:
t = time.time()
scan_stats, profiles = prep.stage_scans(cfg, src, kept)
TIMINGS["scans"] = round(time.time() - t, 1)

## Images

Kept frames' PNGs copied into shards (images from the starter are used in place).

In [ ]:
t = time.time()
prep.stage_images(cfg, src, kept)
index = prep.stage_frame_index(cfg, index, scan_stats)
TIMINGS["images"] = round(time.time() - t, 1)

## Report

In [ ]:
from IPython.display import Image, Markdown, display

t = time.time()
report = prep.stage_report(cfg, found, index, scan_stats, profiles, src)
TIMINGS["report"] = round(time.time() - t, 1)

display(Markdown("\n".join(l for l in report.read_text().splitlines() if not l.startswith("!["))))
for png in sorted(report.parent.glob("*.png")):
    print(png.name)
    display(Image(filename=str(png)))

## Dev sample (drive 0000 only)

In [ ]:
if DEV_FRAMES:
    t = time.time()
    prep.stage_dev_sample(cfg, found, index, src)
    TIMINGS["dev_sample"] = round(time.time() - t, 1)
else:
    print("no dev sample for this drive")

## Manifest

In [ ]:
prep.stage_manifest(cfg, found, index, sources, src["velodyne_read"], TIMINGS)
sh(f'du -sh "{OUT_DIR}"/* | sort -h')

## After the run

- `/kaggle/working/kitti360_supplement` is this notebook's output. Turn it into a private dataset named after the
  drive, e.g. `kitti360-d0009-supplement`, so later notebooks can attach it.
- Send the summary of `report/report.md` back for review.